In [ ]:
import os, glob, numpy as np, trimesh
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# -----------------------------
# Dataset loader (ModelNet10)
# -----------------------------
class ModelNetDataset(Dataset):
    def __init__(self, root, split="train", num_points=2048):
        self.files = glob.glob(os.path.join(root, "*", split, "*.off"))
        self.num_points = num_points
        self.classes = sorted(os.listdir(root))
        self.class_map = {cls: i for i, cls in enumerate(self.classes)}

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        mesh = trimesh.load(self.files[idx])
        points = mesh.sample(self.num_points)
        label = self.class_map[self.files[idx].split("/")[-3]]
        return torch.tensor(points, dtype=torch.float32), label

# -----------------------------
# Orthogonal Regularizer
# -----------------------------
def feature_transform_regularizer(trans):
    d = trans.size()[1]
    I = torch.eye(d, device=trans.device)
    loss = torch.mean(torch.norm(torch.bmm(trans, trans.transpose(2,1)) - I, dim=(1,2)))
    return loss

# -----------------------------
# T-Net (STN)
# -----------------------------
class TNet(nn.Module):
    def __init__(self, k=3):
        super().__init__()
        self.conv1 = nn.Conv1d(k, 64, 1)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.conv3 = nn.Conv1d(128, 1024, 1)
        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, k*k)

        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(1024)
        self.bn4 = nn.BatchNorm1d(512)
        self.bn5 = nn.BatchNorm1d(256)

    def forward(self, x):
        B = x.size(0)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = torch.max(x, 2)[0]
        x = F.relu(self.bn4(self.fc1(x)))
        x = F.relu(self.bn5(self.fc2(x)))
        x = self.fc3(x)

        iden = torch.eye(int(np.sqrt(x.size(1))), device=x.device).view(1, -1).repeat(B, 1)
        x = x + iden
        x = x.view(-1, int(np.sqrt(x.size(1))), int(np.sqrt(x.size(1))))
        return x

# -----------------------------
# PointNet Encoder
# -----------------------------
class PointNetEncoder(nn.Module):
    def __init__(self, global_feat=True, feature_transform=False):
        super().__init__()
        self.stn = TNet(k=3)
        self.conv1 = nn.Conv1d(3, 64, 1)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.conv3 = nn.Conv1d(128, 1024, 1)
        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(1024)
        self.global_feat = global_feat
        self.feature_transform = feature_transform
        if self.feature_transform:
            self.fstn = TNet(k=64)

    def forward(self, x):
        B, N, _ = x.size()
        x = x.transpose(2,1)  # (B,3,N)
        trans = self.stn(x)
        x = torch.bmm(trans, x)

        x = F.relu(self.bn1(self.conv1(x)))

        if self.feature_transform:
            trans_feat = self.fstn(x)
            x = torch.bmm(trans_feat, x)
        else:
            trans_feat = None

        x = F.relu(self.bn2(self.conv2(x)))
        x = self.bn3(self.conv3(x))
        x = torch.max(x, 2)[0]

        return x, trans_feat

# -----------------------------
# PointNet Classifier
# -----------------------------
class PointNetCls(nn.Module):
    def __init__(self, k=10, feature_transform=False):
        super().__init__()
        self.feat = PointNetEncoder(global_feat=True, feature_transform=feature_transform)
        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, k)
        self.bn1 = nn.BatchNorm1d(512)
        self.bn2 = nn.BatchNorm1d(256)
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x):
        x, trans_feat = self.feat(x)
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.dropout(self.fc2(x))))
        x = self.fc3(x)
        return F.log_softmax(x, dim=1), trans_feat

# -----------------------------
# Loss with regularizer
# -----------------------------
class PointNetLoss(nn.Module):
    def __init__(self, mat_diff_loss_scale=0.001):
        super().__init__()
        self.mat_diff_loss_scale = mat_diff_loss_scale

    def forward(self, pred, target, trans_feat):
        loss = F.nll_loss(pred, target)
        if trans_feat is not None:
            loss += feature_transform_regularizer(trans_feat) * self.mat_diff_loss_scale
        return loss
